# 01 · Exploratory data analysis

**Goal:** understand the data well enough to design features (P2) and avoid validation mistakes (P3).

Questions this notebook answers:
1. What does the data look like (volume, tables, stakes, players)?
2. What do the labels cover, and what are the traps (unknown vs negative, label overlap)?
3. How much do pairs play together, in development vs evaluation?
4. What makes the planted **evidence hands** different from ordinary shared hands?
5. How do the pair members *act toward each other* in evidence hands, for each behavior family?
6. What does each family look like hand by hand?

Column semantics are documented in `docs/data_semantics.md`.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from suspoker.config import RAW_DIR
from suspoker.handview import format_hand

pl.Config(tbl_rows=30, tbl_cols=20, tbl_width_chars=180, fmt_str_lengths=60)
FAMILIES = ["directed_transfer", "soft_play", "coordinated_isolation"]

hands = pl.read_parquet(RAW_DIR / "hands.parquet")
seats_lf = pl.scan_parquet(RAW_DIR / "seats.parquet")
actions_lf = pl.scan_parquet(RAW_DIR / "actions.parquet")
labels = pl.read_csv(RAW_DIR / "development_labels.csv")
evidence = pl.read_csv(RAW_DIR / "development_evidence.csv")
eval_pairs = pl.read_csv(RAW_DIR / "evaluation_pairs.csv")
player_table = (seats_lf.select("hand_id", "player_id").unique("player_id")
                .collect().join(hands.select("hand_id", "table_id"), on="hand_id").drop("hand_id"))

## 1. Volume, tables and stakes

In [ ]:
print(hands.group_by("phase").len())
per_table = hands.group_by("table_id").agg(pl.len().alias("hands"), pl.col("big_blind").n_unique().alias("n_stakes"),
                                           pl.col("big_blind").first().alias("bb"))
print(per_table.select(pl.col("hands").min().alias("min_hands"), pl.col("hands").median().alias("median_hands"),
                       pl.col("hands").max().alias("max_hands"), pl.col("n_stakes").max().alias("max_stakes_per_table")))
print(per_table.group_by("bb").len().sort("bb"))
print("players per table:", player_table.group_by("table_id").len()["len"].unique().to_list())

In [ ]:
player_hands = seats_lf.group_by("player_id").len().collect()["len"]
stacks_bb = (seats_lf.select("hand_id", "starting_stack").collect()
             .join(hands.select("hand_id", "big_blind"), on="hand_id")
             .select((pl.col("starting_stack") / pl.col("big_blind")).alias("bb"))["bb"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(player_hands, bins=60); ax[0].set_title("hands played per player")
ax[1].hist(stacks_bb.clip(0, 400), bins=80); ax[1].set_title("starting stack (big blinds, clipped at 400)")
plt.show()
print(stacks_bb.describe())

## 2. Labels: coverage and traps

In [ ]:
print(labels.group_by("label_status", "behavior_family").len().sort("len", descending=True))
lab = (labels.join(player_table.rename({"player_id": "player_1", "table_id": "t1"}), on="player_1")
             .join(player_table.rename({"player_id": "player_2", "table_id": "t2"}), on="player_2"))
print("pairs whose players sit at different tables:", (lab["t1"] != lab["t2"]).sum())
print("tables with >=1 labeled pair:", lab["t1"].n_unique(), "of", hands["table_id"].n_unique())
print("positives per table:")
print(lab.filter(pl.col("label") == 1).group_by("t1").len()["len"].value_counts().sort("len"))

In [ ]:
# Do players appear in several labeled pairs? (rings / hubs would matter for CV grouping)
long = pl.concat([labels.select("pair_id", "label", "behavior_family", pl.col("player_1").alias("player")),
                  labels.select("pair_id", "label", "behavior_family", pl.col("player_2").alias("player"))])
deg = long.group_by("player").agg(pl.len().alias("n_pairs"), pl.col("label").sum().alias("n_pos"))
print("labeled pairs per player:", deg["n_pairs"].value_counts().sort("n_pairs").rows())
print("positive pairs per positive player:", deg.filter(pl.col("n_pos") > 0)["n_pos"].value_counts().sort("n_pos").rows())
fam_per_player = long.filter(pl.col("label") == 1).group_by("player").agg(pl.col("behavior_family").n_unique())
print("positive players with >1 family:", (fam_per_player["behavior_family"] > 1).sum())

pos_players = set(long.filter(pl.col("label") == 1)["player"])
all_labeled = set(long["player"])
ep = pl.concat([eval_pairs["player_1"], eval_pairs["player_2"]])
print("eval pairs containing a labeled-POSITIVE player:", eval_pairs.filter(
    pl.col("player_1").is_in(pos_players) | pl.col("player_2").is_in(pos_players)).height)
print("eval pairs containing a labeled-NEGATIVE-only player:", eval_pairs.filter(
    pl.col("player_1").is_in(all_labeled - pos_players) | pl.col("player_2").is_in(all_labeled - pos_players)).height)
print("eval tables:", eval_pairs.join(player_table.rename({"player_id": "player_1"}), on="player_1")["table_id"].n_unique())

Unlisted development pairs are **unknown**, not negative. With 30 players per table there are 435 pairs per table and 174,000 in total, and only 1,860 are labeled.

## 3. Exposure: how many hands do pairs share?

In [ ]:
labeled_players = list(all_labeled)
s_dev = (seats_lf.filter(pl.col("player_id").is_in(labeled_players))
         .select("hand_id", "player_id").collect()
         .join(hands.select("hand_id", "phase", "started_at"), on="hand_id"))
pair_keys = labels.select("pair_id", "player_1", "player_2", "label", "behavior_family")
shared = (s_dev.rename({"player_id": "player_1"})
          .join(s_dev.select("hand_id", pl.col("player_id").alias("player_2")), on="hand_id")
          .join(pair_keys, on=["player_1", "player_2"]))
print("rows (pair x shared hand):", shared.height)
exp = shared.group_by("pair_id", "phase").len().pivot(on="phase", index="pair_id", values="len").fill_null(0)
exp = exp.join(pair_keys, on="pair_id")
print(exp.group_by("label").agg(pl.col("development").median().alias("dev_median"),
                                 pl.col("evaluation").median().alias("eval_median")))
print("eval_pairs shared_hands:", eval_pairs["shared_hands"].describe().rows())
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(exp["development"], bins=60, alpha=.6, label="labeled pairs, development period", density=True)
ax.hist(eval_pairs["shared_hands"], bins=60, alpha=.6, label="evaluation pairs, evaluation period", density=True)
ax.legend(); ax.set_title("shared hands per pair"); plt.show()

In [ ]:
from suspoker.metric import average_precision

y = exp["label"].to_numpy()
print("base rate:", y.mean().round(3))
print("AP of dev shared-hand count alone (B0-like):", round(average_precision(y, exp["development"].to_numpy().astype(float)), 3))

## 4. Evidence hands vs ordinary shared hands

For each positive pair, compare its **planted evidence hands** with its **other shared development hands**.

In [ ]:
pos_shared = shared.filter((pl.col("label") == 1) & (pl.col("phase") == "development"))
ev_set = evidence.select("pair_id", "hand_id", pl.lit(True).alias("is_evidence"))
pos_shared = pos_shared.join(ev_set, on=["pair_id", "hand_id"], how="left").with_columns(pl.col("is_evidence").fill_null(False))
print("evidence hands found among shared dev hands:", pos_shared["is_evidence"].sum(), "of", evidence.height)

# per-hand context for the pair: both seats' results
needed = pos_shared.select("hand_id").unique()
s_need = seats_lf.join(needed.lazy(), on="hand_id", how="semi").collect()
hctx = (pos_shared
    .join(s_need.select("hand_id", pl.col("player_id").alias("player_1"), pl.col("net_chips").alias("net_1"),
                        pl.col("folded").alias("fold_1"), pl.col("went_to_showdown").alias("sd_1"),
                        pl.col("total_contribution").alias("in_1")), on=["hand_id", "player_1"])
    .join(s_need.select("hand_id", pl.col("player_id").alias("player_2"), pl.col("net_chips").alias("net_2"),
                        pl.col("folded").alias("fold_2"), pl.col("went_to_showdown").alias("sd_2"),
                        pl.col("total_contribution").alias("in_2")), on=["hand_id", "player_2"])
    .join(hands.select("hand_id", "big_blind", "final_pot", "players_at_showdown", "board_cards"), on="hand_id")
    .with_columns(
        (pl.col("final_pot") / pl.col("big_blind")).alias("pot_bb"),
        pl.when(pl.col("board_cards") == "").then(0).otherwise(pl.col("board_cards").str.split(" ").list.len()).alias("board_n"),
        ((pl.col("in_1") > 0) & (pl.col("in_2") > 0)).alias("both_put_in"),
        (pl.col("sd_1") & pl.col("sd_2")).alias("both_showdown"),
        ((pl.col("net_1") - pl.col("net_2")).abs() / pl.col("big_blind") / 2).alias("swing_bb"),
        (((pl.col("net_1") > 0) & (pl.col("net_2") < 0)) | ((pl.col("net_2") > 0) & (pl.col("net_1") < 0))).alias("one_wins_other_loses"),
    ))
summary = (hctx.group_by("behavior_family", "is_evidence")
    .agg(pl.len().alias("n"),
         pl.col("both_put_in").mean().round(3), pl.col("both_showdown").mean().round(3),
         pl.col("one_wins_other_loses").mean().round(3), pl.col("pot_bb").median().round(1).alias("median_pot_bb"),
         pl.col("swing_bb").median().round(1).alias("median_swing_bb"), pl.col("board_n").mean().round(2).alias("mean_board_cards"),
         (pl.col("fold_1") | pl.col("fold_2")).mean().round(3).alias("someone_of_pair_folded"))
    .sort("behavior_family", "is_evidence"))
summary

## 5. How do pair members act toward each other?

For every action by a pair member, find the **last aggressor** before it (the player whose bet, raise or all-in set the current price). Then compare how a member responds when the aggressor is their **partner** vs **someone else**.

In [ ]:
sample_hands = (pos_shared.filter(~pl.col("is_evidence")).group_by("pair_id").head(40)
                .select("pair_id", "hand_id", "is_evidence"))
ev_hands = pos_shared.filter(pl.col("is_evidence")).select("pair_id", "hand_id", "is_evidence")
neg_hands = (shared.filter((pl.col("label") == 0) & (pl.col("phase") == "development"))
             .group_by("pair_id").head(15).select("pair_id", "hand_id", pl.lit(False).alias("is_evidence")))
study = pl.concat([ev_hands, sample_hands, neg_hands]).join(pair_keys, on="pair_id")
acts = (actions_lf.join(study.select("hand_id").unique().lazy(), on="hand_id", how="semi")
        .collect().sort("hand_id", "action_no"))
acts = acts.with_columns(
    pl.when(pl.col("action").is_in(["bet", "raise"]) | ((pl.col("action") == "all_in") & (pl.col("amount") > pl.col("to_call"))))
      .then(pl.col("player_id")).alias("aggr")
).with_columns(pl.col("aggr").shift(1).forward_fill().over("hand_id").alias("last_aggr"))

pa = (study.join(acts, on="hand_id")
      .filter((pl.col("player_id") == pl.col("player_1")) | (pl.col("player_id") == pl.col("player_2")))
      .with_columns(pl.when(pl.col("player_id") == pl.col("player_1")).then(pl.col("player_2"))
                      .otherwise(pl.col("player_1")).alias("partner"))
      .with_columns(pl.when(pl.col("last_aggr").is_null()).then(pl.lit("no bet yet"))
                      .when(pl.col("last_aggr") == pl.col("partner")).then(pl.lit("facing PARTNER"))
                      .otherwise(pl.lit("facing other")).alias("situation"),
                    pl.when(pl.col("label") == 0).then(pl.lit("negative pair"))
                      .when(pl.col("is_evidence")).then(pl.lit("EVIDENCE hand"))
                      .otherwise(pl.lit("other shared hand")).alias("hand_kind")))
resp = (pa.filter(pl.col("situation") != "no bet yet")
        .group_by("behavior_family", "hand_kind", "situation")
        .agg(pl.len().alias("n"),
             (pl.col("action") == "fold").mean().round(3).alias("fold"),
             (pl.col("action") == "call").mean().round(3).alias("call"),
             pl.col("action").is_in(["raise", "all_in"]).mean().round(3).alias("raise/all-in"))
        .sort("behavior_family", "hand_kind", "situation"))
resp

In [ ]:
# Aggression when the partner is still in the hand vs not (open bets/raises when no one has bet yet)
opening = (pa.filter(pl.col("situation") == "no bet yet")
           .group_by("behavior_family", "hand_kind")
           .agg(pl.len().alias("n"), pl.col("action").is_in(["bet", "raise", "all_in"]).mean().round(3).alias("bet_when_unopened"),
                (pl.col("action") == "check").mean().round(3).alias("check_when_unopened"))
           .sort("behavior_family", "hand_kind"))
opening

## 6. Walkthroughs: three pairs per family

In [ ]:
examples = []
for fam in FAMILIES:
    pairs = labels.filter(pl.col("behavior_family") == fam).sort("pair_id").head(3)
    for p in pairs.iter_rows(named=True):
        hid = evidence.filter((pl.col("pair_id") == p["pair_id"]) & (pl.col("evidence_rank") == 1))["hand_id"].item()
        examples.append((fam, p, hid))
ids = [e[2] for e in examples]
s_ex = seats_lf.filter(pl.col("hand_id").is_in(ids)).collect()
a_ex = actions_lf.filter(pl.col("hand_id").is_in(ids)).collect()
h_ex = hands.filter(pl.col("hand_id").is_in(ids))
for fam, p, hid in examples:
    print("=" * 110)
    print(f"{fam.upper()} | pair {p['pair_id']} | P1={p['player_1'][:7]} P2={p['player_2'][:7]} | evidence rank 1")
    print(format_hand(hid, h_ex, s_ex, a_ex, highlight={p["player_1"]: "P1", p["player_2"]: "P2"}))

## 7. Takeaways (first run, 2026-10-04)

**Structure and validation**
- There are 400 tables × 5,000 hands, each with a fixed stake and 30 players. Labels never cross tables, and 397/400 tables have at least one labeled pair.
- Players appear in up to 4 labeled pairs, 49 positive players are in 2–3 positive pairs (hubs/rings), and 30 positive players span more than one family. **This confirms grouped CV by `table_id`**: splitting by pair would leak a player's behavior across folds.
- No evaluation pair contains a labeled-positive player (as promised), but **42,539 evaluation pairs contain a labeled-negative player**. Negatives are not fully held out, but positives are.

**Exposure is not a shortcut**
- Positive and negative pairs share the same number of hands (median ~110 in development). Ranking by shared hands gives **AP 0.204 at a 0.20 base rate**, i.e. no signal. The generator controls for exposure.
- The evaluation period is shorter: labeled pairs share a median of 65–66 evaluation hands vs ~110 development hands. **Features must be rates, shrunk toward the player's baseline.**

**What evidence hands look like (vs the pair's other shared hands)**
| | directed_transfer | soft_play | coordinated_isolation |
|---|---|---|---|
| both pair members put chips in | **100%** (vs 34%) | **99%** (vs 39%) | **71%** (vs 36%) |
| one member wins, the other loses | **100%** | 84% | 44% |
| median pot | **34.5 bb** (vs 10.5) | 17 bb | 18 bb |
| both reach showdown | 25% | **35%** | 3% |

**How members respond to each other** (response rate when the last aggressor is the partner vs someone else)
- **Directed transfer:** facing the partner they *call* 46% (vs 7% facing others) and fold 21%. Facing others they almost never fold (0.6%), because they are in the hand to transfer. Chip dumping looks like paying the partner off with weak hands, and folding a good hand to the partner's bet.
- **Soft play:** facing the partner they *raise* only **1.4%** (vs 10.7% facing others). They check far more often when unopened (25% vs 12%). Members avoid aggression toward each other.
- **Coordinated isolation:** members bet or raise **97.5%** of the time when unopened and re-raise the partner 31% of the time (the squeeze), while third players fold.
- **The signals leak outside the 5 listed evidence hands.** In positive pairs' *other* shared hands, directed-transfer members still call the partner 40% of the time (vs 21% for negative pairs), and isolation members raise the partner 21% (vs 10%). Coordination is episodic, and unlisted hands are also manipulated, so pair-level rates will carry signal.

**Feature design consequences (P2)**
1. **Partner-conditioned response rates** (fold/call/raise facing the partner vs facing others) with shrinkage. This is the core lift signal.
2. **Strength-aware versions:** fold *while ahead* of the partner, call/raise *while behind* (hole cards are known). The walkthroughs show the folded-best-hand and paid-off-with-air patterns directly.
3. **Unopened aggression** and **squeeze sequences** (partner bets, member raises, third player folds) for isolation.
4. **Pot-size-weighted chip flow** between the members, in bb, for directed transfer.
5. Hand-level versions of all of the above for the **evidence ranker** (M4), since evidence hands are extreme on exactly these axes.